# Citation Index API: v0.4.2 executable end-to-end guide

This notebook is both a tutorial and a final smoke test for a deployment. It runs every supported workflow:

1. upload a PDF to the external **MinerU** extractor,
2. ask **Qwen3.6** to identify the bibliography entries,
3. ask **Qwen3.6** to convert those entries into structured records,
4. link a bounded sample of the parsed citations to external scholarly indexes,
5. run the **GROBID** path (PDF → raw + parsed references + citing sentences) in one job, and
6. classify the citing sentences' intent (method / background / result) with the external **CEC** classifier.

Each API operation is asynchronous: submission returns a `job_id`, the client polls `/jobs/{job_id}/status`, and the result is read from `/jobs/{job_id}`. Every route except `/health` requires an `X-API-Key` header when the server sets `API_KEY`. The notebook intentionally fails fast if a service is unavailable or a stage returns an invalid result.

> The combined LLM route `/process/references` is not enabled, so steps 3–5 compose the LLM stages explicitly. `/process/references/grobid` is the one-call GROBID alternative.

## Goal

A successful top-to-bottom run proves all of the following:

- the API, Redis, storage, and RQ workers are available, and the API key is enforced;
- the workers can reach MinerU, GROBID, and the CEC classifier;
- prompt files are present in the image;
- Qwen returns answer content and valid structured JSON;
- the citation-linking queue returns the requested index IDs and DOIs when found, and reports a failing source per link instead of failing the job;
- GROBID returns index-aligned raw references, parsed references, and citing contexts; and
- job failures, missing job IDs, and invalid requests are surfaced as errors rather than empty successes.

## Setup

Start MinerU in one terminal. This command remains attached while the port-forward is active:

```bash
oc port-forward service/mineru-api 8000:8000
```

Start the Docker stack in a second terminal from the repository root. Compose refuses to start the API unless `API_KEY` is set in `.env`; GROBID and CEC are external services: set `GROBID_ENDPOINT` in `.env` (the compose default `http://grobid:8070` has no matching service) and, if needed, `CEC_ENDPOINT`:

```bash
docker compose up -d --build --scale worker-llm=1 \
  redis api worker-default worker-llm worker-linking
docker compose ps
```

The Citation Index API is published on host port **8001** because port **8000** is reserved for MinerU. Docker Compose reads the Qwen endpoint and all keys from `.env`; never put credentials in this notebook.

In [1]:
import json
import os
import time
import uuid
from pathlib import Path

import requests

### Configuration

Defaults target the staging deployment, a small repository fixture, and Matilda linking for at most three extracted references. Override them through environment variables when needed. Set `CITATION_INDEX_LINKING_TARGETS` to `all` or a comma-separated list of `openalex`, `matilda`, `wikidata`, and `opencitations`. `CITATION_INDEX_API_KEY` must match the server's `API_KEY`; leave it empty only against a server with auth disabled.

```bash
export CITATION_INDEX_API=http://localhost:8001
export CITATION_INDEX_API_KEY=...            # the server's API_KEY
export CITATION_INDEX_PDF=/absolute/path/to/paper.pdf
export CITATION_INDEX_LINKING_TARGETS=matilda
export CITATION_INDEX_LINKING_LIMIT=3
export CITATION_INDEX_CEC_LIMIT=10
```

In [2]:
API_BASE = os.getenv(
    "CITATION_INDEX_API",
    "https://citation-index-api-graphia-app1-staging.apps.bst2.paas.psnc.pl",
).rstrip("/")
API_KEY = os.getenv("CITATION_INDEX_API_KEY", "")
EXPECTED_VERSION = "0.4.2"
PDF_PATH = Path(
    os.getenv("CITATION_INDEX_PDF", "../benchmarks/excite/all_pdfs/44404.pdf")
).expanduser().resolve()
LINKING_TARGETS = os.getenv("CITATION_INDEX_LINKING_TARGETS", "matilda")
MAX_LINKING_REFERENCES = int(os.getenv("CITATION_INDEX_LINKING_LIMIT", "3"))
MAX_CEC_CITATIONS = int(os.getenv("CITATION_INDEX_CEC_LIMIT", "10"))

POLL_INTERVAL_SECONDS = 2
MAX_JOB_WAIT_SECONDS = 1_800
HTTP_TIMEOUT = (10, 120)  # connect timeout, response timeout

session = requests.Session()
if API_KEY:
    session.headers["X-API-Key"] = API_KEY
print(
    {
        "api": API_BASE,
        "api_key_set": bool(API_KEY),
        "pdf": str(PDF_PATH),
        "pdf_exists": PDF_PATH.is_file(),
        "linking_targets": LINKING_TARGETS,
        "linking_limit": MAX_LINKING_REFERENCES,
        "cec_limit": MAX_CEC_CITATIONS,
    }
)
assert PDF_PATH.is_file(), f"PDF not found: {PDF_PATH}"
assert MAX_LINKING_REFERENCES > 0, "CITATION_INDEX_LINKING_LIMIT must be positive"
assert MAX_CEC_CITATIONS > 0, "CITATION_INDEX_CEC_LIMIT must be positive"

{'api': 'https://citation-index-api-graphia-app1-staging.apps.bst2.paas.psnc.pl', 'api_key_set': True, 'pdf': '/Users/alex/docs/code/Odoma/citation_index/benchmarks/excite/all_pdfs/44404.pdf', 'pdf_exists': True, 'linking_targets': 'matilda', 'linking_limit': 3, 'cec_limit': 10}


## Step 1 — Verify service health and authentication

`GET /health` checks the API's Redis and storage dependencies and is the only route that needs no API key (liveness probes carry none). It does not call MinerU, Qwen, GROBID, or CEC, so those integrations are tested by the later stages.

The second check sends a request without the key to a protected route. A server with `API_KEY` set must answer HTTP 401; the check is skipped when no key is configured locally.

In [3]:
health_response = session.get(f"{API_BASE}/health", timeout=HTTP_TIMEOUT)
health_response.raise_for_status()
health = health_response.json()
print(health)
assert health["status"] == "healthy", health
assert health["redis"] == "ok" and health["storage"] == "ok", health
assert health["version"] == EXPECTED_VERSION, health

if API_KEY:
    unauthenticated = requests.get(
        f"{API_BASE}/jobs/{uuid.uuid4()}/status", timeout=HTTP_TIMEOUT
    )
    print({"status_code_without_key": unauthenticated.status_code})
    assert unauthenticated.status_code == 401, unauthenticated.text
    auth_enforced = True
else:
    print("CITATION_INDEX_API_KEY not set: assuming the server runs without auth")
    auth_enforced = None

{'status': 'healthy', 'redis': 'ok', 'storage': 'ok', 'version': '0.4.2'}


{'status_code_without_key': 401}


## Step 2 — Understand the asynchronous job contract

Submission endpoints return HTTP 200 with a job object such as:

```json
{"job_id": "...", "status": "queued", "created_at": "..."}
```

A job then moves through `queued` and `processing` to either `completed` or `failed`. The helper below prints only state changes, raises with the server error on failure, and enforces a maximum wait.

In [4]:
def response_json(response: requests.Response) -> dict:
    """Raise an informative HTTP error, then return a JSON object."""
    try:
        payload = response.json()
    except requests.JSONDecodeError as exc:
        preview = response.text[:300].replace("\n", " ")
        raise RuntimeError(
            f"Expected JSON from {response.request.method} {response.url}; "
            f"HTTP {response.status_code}, body={preview!r}"
        ) from exc
    if not response.ok:
        raise RuntimeError(
            f"{response.request.method} {response.url} failed with "
            f"HTTP {response.status_code}: {payload}"
        )
    if not isinstance(payload, dict):
        raise TypeError(f"Expected a JSON object, got {type(payload).__name__}")
    return payload


def wait_for_job(job_id: str, max_wait: int = MAX_JOB_WAIT_SECONDS) -> dict:
    """Poll a job to a terminal state and return its persisted result."""
    started = time.monotonic()
    last_state = None

    while time.monotonic() - started < max_wait:
        status_response = session.get(
            f"{API_BASE}/jobs/{job_id}/status", timeout=HTTP_TIMEOUT
        )
        status_payload = response_json(status_response)
        state = status_payload["status"]

        if state != last_state:
            elapsed = time.monotonic() - started
            stage = status_payload.get("current_stage")
            print(f"[{elapsed:6.1f}s] {job_id}: {state} (stage={stage})")
            last_state = state

        if state == "completed":
            return response_json(
                session.get(f"{API_BASE}/jobs/{job_id}", timeout=HTTP_TIMEOUT)
            )
        if state == "failed":
            raise RuntimeError(
                f"Job {job_id} failed: {status_payload.get('error', 'unknown error')}"
            )

        time.sleep(POLL_INTERVAL_SECONDS)

    raise TimeoutError(f"Job {job_id} did not finish within {max_wait}s")

## Step 3 — Extract Markdown with MinerU

`POST /extract/text` accepts a PDF as multipart form data. Supported extractor names are `pymupdf`, `mineru`, and `grobid`; any other value is rejected with HTTP 422 before a job is queued. This test selects `mineru`, which makes a real call to the configured MinerU service.

> MinerU (`vlm-auto-engine`) can drop footnotes on footnote-style humanities papers; prefer `pymupdf` there.

In [5]:
with PDF_PATH.open("rb") as pdf_file:
    submit_text_response = session.post(
        f"{API_BASE}/extract/text",
        params={"extractor": "mineru", "markdown": "true"},
        files={"file": (PDF_PATH.name, pdf_file, "application/pdf")},
        timeout=HTTP_TIMEOUT,
    )

text_job = response_json(submit_text_response)
print(text_job)
assert text_job["status"] in {"queued", "processing"}

{'job_id': '24f92fb7-cc0d-40e0-a381-a0d10fe508cc', 'status': 'queued', 'created_at': '2026-11-04T14:11:41.501570', 'message': 'Text extraction job enqueued'}


In [6]:
text_result = wait_for_job(text_job["job_id"])
extracted_text = text_result.get("text", "")

assert text_result.get("extractor") == "mineru", text_result
assert len(extracted_text) > 1_000, "MinerU returned unexpectedly little text"

print(
    {
        "extractor": text_result.get("extractor"),
        "backend": text_result.get("backend"),
        "mineru_version": text_result.get("mineru_version"),
        "text_characters": len(extracted_text),
    }
)
print("\nPreview:\n", extracted_text[:700].strip())

[   0.0s] 24f92fb7-cc0d-40e0-a381-a0d10fe508cc: queued (stage=None)


[   2.1s] 24f92fb7-cc0d-40e0-a381-a0d10fe508cc: processing (stage=text_extraction)


[   8.3s] 24f92fb7-cc0d-40e0-a381-a0d10fe508cc: completed (stage=text_extraction)
{'extractor': 'mineru', 'backend': None, 'mineru_version': None, 'text_characters': 32678}

Preview:
 # BRENNPUNKT LATEINAMERIKA

POLITIK · WIRTSCHAFT · GESELLSCHAFT

INSTITUT FÜR IBEROAMERIKA-KUNDE HAMBURG

Nummer 12

30. Juni 2000

ISSN 1437-6148

# Die politische Krise in Peru: Festsetzung des Fujimorismo und Polarisierung des Landes

Andreas Steinhauf

Von den umstrittensten und schmutzigsten Wahlen in der Geschichte Perus ist die Rede. Am 28. Mai 2000 wurde der amtierende Präsident Alberto Fujimori von der Obersten Wahlbehörde zum Sieger der Stichwahl erklärt, zu der sein Kontrahent Alejandro Toledo bereits nicht mehr angetreten war und stattdessen die Bevölkerung zum Boykott aufgerufen hatte. Damit wird er am 28. Juli, wenn verfassungsgemäß der neue Präsident vereidigt wird, seine nunm


## Step 4 — Extract raw bibliography entries with Qwen

`POST /extract/references` with a JSON `text` body sends the Markdown to the configured medium-intelligence model (`Qwen3.6-27B` by default). `method` is `full_text` (default) or `semantic`. Temperature `0.0` makes this validation run as deterministic as the serving stack permits.

The same route also accepts a multipart PDF with `method=grobid` (see Step 7). Mixing the two — a PDF without `method=grobid`, or `include_parsed` / `include_context` on a JSON body — returns HTTP 400.

In [7]:
submit_extraction_response = session.post(
    f"{API_BASE}/extract/references",
    params={"method": "full_text", "temperature": 0.0},
    json={"text": extracted_text},
    timeout=HTTP_TIMEOUT,
)

extraction_job = response_json(submit_extraction_response)
print(extraction_job)

{'job_id': '5d73afbd-ae07-426e-ae75-874720d35f68', 'status': 'queued', 'created_at': '2026-11-04T14:11:50.181360', 'message': 'Reference extraction job enqueued'}


In [8]:
extraction_result = wait_for_job(extraction_job["job_id"])
raw_references = extraction_result.get("references", [])

assert raw_references, "Qwen returned no references for a document with a bibliography"
assert extraction_result.get("count") == len(raw_references)
assert all(isinstance(item, str) and item.strip() for item in raw_references)

print(f"Extracted {len(raw_references)} bibliography entries:")
for index, reference in enumerate(raw_references[:10], start=1):
    print(f"{index:>2}. {reference[:300]}")

[   0.0s] 5d73afbd-ae07-426e-ae75-874720d35f68: queued (stage=None)


[   2.1s] 5d73afbd-ae07-426e-ae75-874720d35f68: processing (stage=reference_extraction)


[   8.3s] 5d73afbd-ae07-426e-ae75-874720d35f68: completed (stage=reference_extraction)
Extracted 8 bibliography entries:
 1. Resumen Semanal, DESCO, Lima: http://www.desco.org.pe/rs-in.HTM
 2. Que Hacer, DESCO, Lima: http://www.desco.org.pe/qh/qh-in.htm#OH
 3. Caretas: http://www.caretas.com.pe/
 4. El Comercio: http://www.elcomercioperu.com
 5. Eigenen Beobachtungen vom 8. - 16 Juni 2000 in Lima
 6. Ingolf Dietrich. Die Koka- und Kokainwirtschaft Perus. Frankfurt/M.: Vervuert 1998, 314 S., ISBN 3-89354-247-7, Band 48
 7. Peter Thiery. Transformation in Chile. Institutioneller Wandel, Entwicklung und Demokratie 1973-1996. Frankfurt/M.: Vervuert 2000, ca. 354 S., Band 51
 8. Judith Schultz. Präsidentielle Demokratien in Lateinamerika. Eine Untersuchung der präsidentiellen Regierungssysteme von Costa Rica und Venezuela. Frankfurt/M.: Vervuert 2000, ca. 490 S., Band 52


## Step 5 — Parse the entries into structured records

`POST /parse/references` accepts a JSON list of strings. With `parser=llm`, Qwen returns fields such as authors, title, year, publication venue, volume, issue, and identifiers when they are present in the source citation. `parser=grobid` is the alternative; other values return HTTP 422. A custom `prompt_name` must name an existing file under `prompts/` (HTTP 400 otherwise).

In [9]:
submit_parse_response = session.post(
    f"{API_BASE}/parse/references",
    params={"parser": "llm", "temperature": 0.0},
    json={"references": raw_references},
    timeout=HTTP_TIMEOUT,
)

parse_job = response_json(submit_parse_response)
print(parse_job)

{'job_id': 'd9865e01-4140-4046-95f0-8313d57a4c55', 'status': 'queued', 'created_at': '2026-11-04T14:11:58.673216', 'message': 'Reference parsing job enqueued'}


In [10]:
parse_result = wait_for_job(parse_job["job_id"])
structured_references = parse_result.get("references", [])

assert parse_result.get("parser") == "llm", parse_result
assert parse_result.get("count") == len(structured_references)
assert len(structured_references) == len(raw_references)
assert all(isinstance(item, dict) for item in structured_references)

# Counts alone cannot detect a field-level regression: one record per input string is
# returned even when every title is null. Require most references to carry a title.
MIN_TITLED_SHARE = 0.8

untitled = [
    reference
    for reference in structured_references
    if not (reference.get("full_title") or reference.get("journal_title"))
]
titled_share = 1 - len(untitled) / len(structured_references)
assert titled_share >= MIN_TITLED_SHARE, (
    f"only {titled_share:.0%} of references carry a title, expected at least "
    f"{MIN_TITLED_SHARE:.0%}; untitled entries: {untitled}"
)

print(f"Parsed {len(structured_references)} structured references, {titled_share:.0%} titled.")
print(json.dumps(structured_references[:2], indent=2, ensure_ascii=False)[:4_000])

[   0.0s] d9865e01-4140-4046-95f0-8313d57a4c55: queued (stage=None)


[   2.1s] d9865e01-4140-4046-95f0-8313d57a4c55: completed (stage=reference_parsing)
Parsed 8 structured references, 100% titled.
[
  {
    "full_title": "Resumen Semanal",
    "journal_title": "DESCO",
    "authors": null,
    "editors": null,
    "publisher": "DESCO",
    "translator": null,
    "publication_place": "Lima",
    "publication_year": null,
    "publication_date_raw": null,
    "identifiers": [
      {
        "scheme": "URL",
        "value": "http://www.desco.org.pe/rs-in.HTM",
        "normalized": null
      }
    ],
    "ref_type": "article",
    "raw": {},
    "volume": null,
    "issue": null,
    "pages": null,
    "cited_range": null,
    "footnote_number": null
  },
  {
    "full_title": "Que Hacer",
    "journal_title": "DESCO",
    "authors": null,
    "editors": null,
    "publisher": "DESCO",
    "translator": null,
    "publication_place": "Lima",
    "publication_year": null,
    "publication_date_raw": null,
    "identifiers": [
      {
        "scheme": 

## Step 6 — Link citations to scholarly indexes

`POST /link/references` accepts a parsed reference object with required `full_title`, non-empty `authors`, and integer `publication_year` fields. With `batched=true`, `reference` must be a non-empty JSON array of these objects. Raw strings and objects missing a field are rejected with HTTP 422; an array without `batched=true` (or the reverse) and an unknown target are rejected with HTTP 400. Candidates are searched using parsed fields and filtered with `custom_match()`: title similarity ≥90 and either first-author similarity ≥70 or year within ±1.

The `target` query parameter accepts `openalex`, `matilda`, `wikidata`, `opencitations`, `all`, or a comma-separated list. OpenCitations returns OMIDs (`br/...`) and is the slowest target (SPARQL fallback, several seconds per reference). One failing source does not fail the job: its link comes back as `{"id": null, "doi": null, "error": "..."}`. The job fails only when every lookup fails.

This executable check takes the structured records from Step 5, selects records with the required title, first-author name, and year, and submits only those three fields. It links at most `MAX_LINKING_REFERENCES` eligible citations to keep live external requests bounded. A target can legitimately return `null` for both fields when it cannot make a confident match. The default target is Matilda; it needs the deployment's Matilda credentials on the linking worker. OpenAlex is not the default because shared OpenAlex quotas can return HTTP 429, which fails a single-target job.

In [11]:
eligible_references = []
for reference in structured_references:
    title = reference.get("full_title")
    authors = reference.get("authors") or []
    year = reference.get("publication_year")
    first_author = authors[0] if authors else None
    author_name = (
        first_author
        if isinstance(first_author, str)
        else (first_author.get("surname") or first_author.get("name"))
        if isinstance(first_author, dict)
        else None
    )
    if (
        isinstance(title, str) and title.strip()
        and isinstance(author_name, str) and author_name.strip()
        and type(year) is int and 1 <= year <= 9999
    ):
        eligible_references.append({
            "full_title": title,
            "authors": [
                {key: value for key, value in author.items() if value is not None}
                if isinstance(author, dict) else author
                for author in authors
            ],
            "publication_year": year,
        })

print(f"Skipped {len(structured_references) - len(eligible_references)} records missing linking fields")
references_to_link = eligible_references[:MAX_LINKING_REFERENCES]
assert references_to_link, "No parsed references have the required title, author, and year"

submit_link_response = session.post(
    f"{API_BASE}/link/references",
    params={"batched": "true", "target": LINKING_TARGETS},
    json={"reference": references_to_link},
    timeout=HTTP_TIMEOUT,
)

link_job = response_json(submit_link_response)
print(link_job)

Skipped 5 records missing linking fields


{'job_id': '1fad3cf1-516b-4049-8af6-bb5ce59291f4', 'status': 'queued', 'created_at': '2026-11-04T14:12:00.973394', 'message': 'Citation linking job enqueued'}


In [12]:
link_result = wait_for_job(link_job["job_id"])
linked_references = link_result.get("results", [])
requested_targets = [
    target.strip().lower()
    for target in LINKING_TARGETS.split(",")
    if target.strip()
]
expected_linking_targets = (
    ["openalex", "matilda", "wikidata", "opencitations"]
    if "all" in requested_targets
    else list(dict.fromkeys(requested_targets))
)

assert link_result.get("count") == len(references_to_link), link_result
assert link_result.get("targets") == expected_linking_targets, link_result
assert len(linked_references) == len(references_to_link), link_result
assert [item.get("reference") for item in linked_references] == references_to_link
for item in linked_references:
    assert list(item.get("links", {})) == expected_linking_targets, item
    for link in item["links"].values():
        assert set(link) in ({"id", "doi"}, {"id", "doi", "error"}), link
        assert link["id"] is None or isinstance(link["id"], str), link
        assert link["doi"] is None or isinstance(link["doi"], str), link

linked_id_count = sum(
    link["id"] is not None
    for item in linked_references
    for link in item["links"].values()
)
source_errors = [
    (target, link["error"])
    for item in linked_references
    for target, link in item["links"].items()
    if link.get("error")
]
if source_errors:
    print(f"WARNING: {len(source_errors)} per-source linking errors: {source_errors}")
print(
    json.dumps(
        {"linked_id_count": linked_id_count, "results": linked_references},
        indent=2,
        ensure_ascii=False,
    )
)

[   0.1s] 1fad3cf1-516b-4049-8af6-bb5ce59291f4: processing (stage=citation_linking)


[  16.7s] 1fad3cf1-516b-4049-8af6-bb5ce59291f4: completed (stage=citation_linking)
{
  "linked_id_count": 2,
  "results": [
    {
      "reference": {
        "full_title": "Die Koka- und Kokainwirtschaft Perus",
        "authors": [
          {
            "first_name": "Ingolf",
            "surname": "Dietrich"
          }
        ],
        "publication_year": 1998
      },
      "links": {
        "matilda": {
          "id": "0a9b236a-ea38-4064-9307-94e5aaf0a014",
          "doi": "10.31819/9783964566904"
        }
      }
    },
    {
      "reference": {
        "full_title": "Transformation in Chile. Institutioneller Wandel, Entwicklung und Demokratie 1973-1996",
        "authors": [
          {
            "first_name": "Peter",
            "surname": "Thiery"
          }
        ],
        "publication_year": 2000
      },
      "links": {
        "matilda": {
          "id": "f33b84ae-55bc-44ba-a786-21ae23231c9f",
          "doi": "10.31819/9783964567734"
        }
      }


## Step 7 — One-call GROBID path with citing contexts

`POST /process/references/grobid` uploads the PDF to GROBID fulltext and returns, in one job on the `default` queue:

- `references` — raw reference strings,
- `parsed_references` — structured records (or `null` when GROBID could not parse an entry), and
- with `include_context=true`, `contexts` — per reference, the sentences that cite it as `{section, citation, marker, ref_id}`.

The three lists are index-aligned. The same result is available from `POST /extract/references?method=grobid&include_parsed=true&include_context=true` with a multipart PDF. No LLM is involved, so this path is much faster than Steps 3–5, but it depends on GROBID's segmentation.

In [13]:
with PDF_PATH.open("rb") as pdf_file:
    submit_grobid_response = session.post(
        f"{API_BASE}/process/references/grobid",
        params={"include_context": "true"},
        files={"file": (PDF_PATH.name, pdf_file, "application/pdf")},
        timeout=HTTP_TIMEOUT,
    )

grobid_job = response_json(submit_grobid_response)
print(grobid_job)

{'job_id': 'c9eedd7e-0dda-4681-95ec-ac103caffa68', 'status': 'queued', 'created_at': '2026-11-04T14:12:17.918713', 'message': 'GROBID reference extraction job enqueued'}


In [14]:
grobid_result = wait_for_job(grobid_job["job_id"])
grobid_references = grobid_result.get("references", [])
grobid_parsed = grobid_result.get("parsed_references", [])
grobid_contexts = grobid_result.get("contexts", [])

assert grobid_result.get("method") == "grobid", grobid_result
assert grobid_references, "GROBID returned no references"
assert grobid_result.get("count") == len(grobid_references)
assert len(grobid_parsed) == len(grobid_references) == len(grobid_contexts)
assert all(item is None or isinstance(item, dict) for item in grobid_parsed)
for reference_contexts in grobid_contexts:
    for context in reference_contexts:
        assert {"section", "citation", "marker", "ref_id"} <= set(context), context

cited_count = sum(bool(reference_contexts) for reference_contexts in grobid_contexts)
print(
    f"GROBID: {len(grobid_references)} references, "
    f"{sum(item is not None for item in grobid_parsed)} parsed, "
    f"{cited_count} with at least one citing sentence"
)
for raw, parsed, reference_contexts in list(
    zip(grobid_references, grobid_parsed, grobid_contexts)
)[:3]:
    print("\n-", raw[:200])
    print("  title:", (parsed or {}).get("full_title"), "| pages:", (parsed or {}).get("pages"))
    for context in reference_contexts[:2]:
        print(f"  cited in [{context['section'] or '?'}]: {context['citation'][:200]}")

[   0.0s] c9eedd7e-0dda-4681-95ec-ac103caffa68: queued (stage=None)


[   2.1s] c9eedd7e-0dda-4681-95ec-ac103caffa68: processing (stage=reference_extraction)


[   4.2s] c9eedd7e-0dda-4681-95ec-ac103caffa68: completed (stage=reference_extraction)
GROBID: 3 references, 3 parsed, 0 with at least one citing sentence

- Quellen: Resumen Semanal, DESCO, Lima: http://www.desco.org.pe/rs-in.HTM Que Hacer, DESCO, Lima: http://www.desco.org.pe/qh/qh-in.htm#QH Caretas: http://www.caretas.com.pe/ El Comercio: http://www.elc
  title: Institut für Iberoamerika-Kunde. Länderschwerpunkte: Peru, Ekuador und Bolivien. Themenschwerpunkte: Informeller Sektor, Migration, soziale Netzwerke und Multikulturalität. E-mail: steinhauf@public.uni | pages: 354

- S., Band 51
  title: None | pages: None

- Judith Schultz Präsidentielle Demokratien in Lateinamerika. Eine Untersuchung der präsidentiellen Regierungssysteme von Costa Rica und Venezuela. Frankfurt/M.: Vervuert 2000, ca. 490 S., Band 52 Impre
  title: den Verbund der Stiftung Deutsches Übersee-Institut. Aufgabe des IIK ist die gegenwartsbezogene Beobachtung und wissenschaftliche Untersuchung der politischen, w

## Step 8 — Classify citation intent with CEC

`POST /classify/citations` takes `{"citations": [{"section": "...", "citation": "..."}, ...]}` and returns one result per sentence, in input order. `mode` is `WS` (with sections), `WoS` (without), or `M` (default: per item, with section when non-empty). GROBID contexts from Step 7 can be posted as-is; extra keys such as `marker` and `ref_id` are ignored.

CEC serves one request at a time and rebuilds its predictor per call, so send a paper's citations in **one** request rather than one per sentence. This check sends at most `MAX_CEC_CITATIONS` sentences. If GROBID resolved no citing sentences in the PDF (common for short briefs without inline markers), one fixed example sentence is sent instead so CEC is always exercised; `cec_source` in the verdict says which input was used.

In [15]:
citations_to_classify = [
    {"section": context["section"], "citation": context["citation"]}
    for reference_contexts in grobid_contexts
    for context in reference_contexts
][:MAX_CEC_CITATIONS]
cec_source = "grobid"
if not citations_to_classify:
    print("No citing sentences found by GROBID; classifying a fixed example sentence")
    citations_to_classify = [
        {"section": "Methods", "citation": "We follow the approach of Smith et al. (2020)."}
    ]
    cec_source = "fallback"

submit_cec_response = session.post(
    f"{API_BASE}/classify/citations",
    params={"mode": "M"},
    json={"citations": citations_to_classify},
    timeout=HTTP_TIMEOUT,
)
cec_job = response_json(submit_cec_response)
print(cec_job)

cec_result = wait_for_job(cec_job["job_id"])
cec_labels = cec_result.get("results", [])
assert cec_result.get("mode") == "M", cec_result
assert cec_result.get("count") == len(citations_to_classify), cec_result
assert len(cec_labels) == len(citations_to_classify), cec_result

for citation, label in zip(citations_to_classify, cec_labels):
    print(f"{json.dumps(label, ensure_ascii=False)[:120]}  <-  {citation['citation'][:120]}")

No citing sentences found by GROBID; classifying a fixed example sentence
{'job_id': '068e4429-94d6-4c3e-b02b-06e37c9055dc', 'status': 'queued', 'created_at': '2026-11-04T14:12:22.310792', 'message': 'Citation classification job enqueued'}
[   0.0s] 068e4429-94d6-4c3e-b02b-06e37c9055dc: queued (stage=None)


[   2.1s] 068e4429-94d6-4c3e-b02b-06e37c9055dc: processing (stage=citation_classification)


[  33.3s] 068e4429-94d6-4c3e-b02b-06e37c9055dc: completed (stage=citation_classification)
{"SECTION": "Methods", "CITATION": "We follow the approach of Smith et al. (2020).", "SCIBERT MET POSITIVE PROBABILITY":  <-  We follow the approach of Smith et al. (2020).


## Checks — verify the error contract

A missing job must return HTTP 404. In production code, use `response_json` or equivalent handling so failed jobs and invalid IDs cannot be mistaken for empty results.

Invalid requests must be rejected before a job is created:

- linking: raw strings and parsed objects missing a required field → 422; an array without `batched=true` or an unknown target → 400;
- unknown `extractor` / `parser` / `method` values → 422;
- `include_context` on a JSON (LLM) extraction request → 400;
- an empty `citations` list for CEC, or an unknown `mode` → 422 / 400.

In [16]:
missing_job_id = str(uuid.uuid4())
missing_response = session.get(
    f"{API_BASE}/jobs/{missing_job_id}/status", timeout=HTTP_TIMEOUT
)
print({"status_code": missing_response.status_code, "body": missing_response.json()})
assert missing_response.status_code == 404

invalid_linking_inputs = ["Smith. A paper. 2024."]
for field in ("full_title", "authors", "publication_year"):
    incomplete_reference = dict(references_to_link[0])
    del incomplete_reference[field]
    invalid_linking_inputs.append(incomplete_reference)

for invalid_reference in invalid_linking_inputs:
    invalid_response = session.post(
        f"{API_BASE}/link/references",
        json={"reference": invalid_reference},
        timeout=HTTP_TIMEOUT,
    )
    assert invalid_response.status_code == 422, invalid_response.text
print("Invalid linking inputs rejected with HTTP 422")

# (method, path, params, json body, expected status)
invalid_requests = [
    ("link: array without batched", "/link/references", {}, {"reference": references_to_link[:1]}, 400),
    ("link: unknown target", "/link/references", {"target": "crossref"}, {"reference": references_to_link[0]}, 400),
    ("extract: unknown method", "/extract/references", {"method": "page_by_page"}, {"text": "x"}, 422),
    ("extract: context without grobid", "/extract/references", {"include_context": "true"}, {"text": "x"}, 400),
    ("parse: unknown parser", "/parse/references", {"parser": "regex"}, {"references": ["x"]}, 422),
    ("classify: empty citations", "/classify/citations", {}, {"citations": []}, 422),
    ("classify: unknown mode", "/classify/citations", {"mode": "XX"}, {"citations": [{"citation": "x"}]}, 400),
]
for label, path, params, body, expected in invalid_requests:
    response = session.post(f"{API_BASE}{path}", params=params, json=body, timeout=HTTP_TIMEOUT)
    assert response.status_code == expected, (label, response.status_code, response.text)
print(f"{len(invalid_requests)} invalid requests rejected with the expected 400/422")

{'status_code': 404, 'body': {'detail': 'Job cf1c339e-1535-4c1c-a31c-2b83bbe1c84d not found'}}


Invalid linking inputs rejected with HTTP 422


7 invalid requests rejected with the expected 400/422


## Checks — final end-to-end verdict

This cell deliberately contains assertions, not just display logic. If it prints `PASS`, the tested PDF completed every supported stage and all required outputs were persisted.

In [17]:
end_to_end_checks = {
    "api_version": health["version"] == EXPECTED_VERSION,
    "api_healthy": health["status"] == "healthy",
    "mineru_text_returned": len(extracted_text) > 1_000,
    "raw_references_returned": len(raw_references) > 0,
    "all_references_parsed": len(structured_references) == len(raw_references),
    "references_carry_titles": titled_share >= MIN_TITLED_SHARE,
    "all_linking_results_returned": len(linked_references) == len(references_to_link),
    "linking_shape_is_valid": all(
        list(item.get("links", {})) == expected_linking_targets
        for item in linked_references
    ),
    "grobid_references_returned": len(grobid_references) > 0,
    "grobid_outputs_aligned": len(grobid_parsed) == len(grobid_references) == len(grobid_contexts),
    "cec_results_aligned": len(cec_labels) == len(citations_to_classify),
    "missing_job_returns_404": missing_response.status_code == 404,
}
assert all(end_to_end_checks.values()), end_to_end_checks

verdict = {
    "verdict": "PASS",
    "version": health["version"],
    "auth_enforced": auth_enforced,
    "pdf": PDF_PATH.name,
    "text_characters": len(extracted_text),
    "raw_reference_count": len(raw_references),
    "structured_reference_count": len(structured_references),
    "titled_share": round(titled_share, 3),
    "linking_reference_count": len(linked_references),
    "linking_targets": expected_linking_targets,
    "linked_id_count": linked_id_count,
    "linking_source_errors": len(source_errors),
    "grobid_reference_count": len(grobid_references),
    "grobid_cited_reference_count": cited_count,
    "cec_classified_count": len(cec_labels),
    "cec_source": cec_source,
    "jobs": {
        "text_extraction": text_job["job_id"],
        "reference_extraction": extraction_job["job_id"],
        "reference_parsing": parse_job["job_id"],
        "citation_linking": link_job["job_id"],
        "grobid_reference_pipeline": grobid_job["job_id"],
        "citation_classification": cec_job["job_id"],
    },
}
print(json.dumps(verdict, indent=2))

{
  "verdict": "PASS",
  "version": "0.4.2",
  "auth_enforced": true,
  "pdf": "44404.pdf",
  "text_characters": 32678,
  "raw_reference_count": 8,
  "structured_reference_count": 8,
  "titled_share": 1.0,
  "linking_reference_count": 3,
  "linking_targets": [
    "matilda"
  ],
  "linked_id_count": 2,
  "linking_source_errors": 0,
  "grobid_reference_count": 3,
  "grobid_cited_reference_count": 0,
  "cec_classified_count": 1,
  "cec_source": "fallback",
  "jobs": {
    "text_extraction": "24f92fb7-cc0d-40e0-a381-a0d10fe508cc",
    "reference_extraction": "5d73afbd-ae07-426e-ae75-874720d35f68",
    "reference_parsing": "d9865e01-4140-4046-95f0-8313d57a4c55",
    "citation_linking": "1fad3cf1-516b-4049-8af6-bb5ce59291f4",
    "grobid_reference_pipeline": "c9eedd7e-0dda-4681-95ec-ac103caffa68",
    "citation_classification": "068e4429-94d6-4c3e-b02b-06e37c9055dc"
  }
}


## Equivalent command-line workflow

Every request except `/health` carries the API key:

```bash
export CITATION_INDEX_API_KEY=...
```

Submit the first stage:

```bash
curl -H "X-API-Key: $CITATION_INDEX_API_KEY" \
  -F "file=@/absolute/path/paper.pdf;type=application/pdf" \
  "http://localhost:8001/extract/text?extractor=mineru&markdown=true"
```

Poll and retrieve any returned job ID:

```bash
curl -H "X-API-Key: $CITATION_INDEX_API_KEY" http://localhost:8001/jobs/JOB_ID/status
curl -H "X-API-Key: $CITATION_INDEX_API_KEY" http://localhost:8001/jobs/JOB_ID
```

Submit a parsed citation batch to more than one linking target:

```bash
curl -X POST -H "X-API-Key: $CITATION_INDEX_API_KEY" \
  "http://localhost:8001/link/references?batched=true&target=matilda,opencitations" \
  -H "Content-Type: application/json" \
  -d '{"reference":[{"full_title":"Deep learning","authors":["LeCun, Y."],"publication_year":2015}]}'
```

Run the GROBID path with citing contexts, then classify sentences:

```bash
curl -H "X-API-Key: $CITATION_INDEX_API_KEY" \
  -F "file=@/absolute/path/paper.pdf;type=application/pdf" \
  "http://localhost:8001/process/references/grobid?include_context=true"

curl -X POST -H "X-API-Key: $CITATION_INDEX_API_KEY" \
  "http://localhost:8001/classify/citations?mode=M" \
  -H "Content-Type: application/json" \
  -d '{"citations":[{"section":"Methods","citation":"We follow the approach of Smith et al. (2020)."}]}'
```

For the other JSON stages, write the previous response to a file or use `jq` to build the next request. Python is usually clearer for the complete chain because extracted document text can be large.

## API quick reference

All routes except `/health` require `X-API-Key` when the server sets `API_KEY`.

| Method | Path | Queue | Input | Result |
|---|---|---|---|---|
| `GET` | `/health` | — | — | service health (no key needed) |
| `POST` | `/extract/text` | `default` | multipart PDF; `extractor=pymupdf\|mineru\|grobid` | Markdown text |
| `POST` | `/extract/references` | `llm-tasks` | `{"text": "..."}`; `method=full_text\|semantic` | raw citation strings |
| `POST` | `/extract/references` | `default` | multipart PDF; `method=grobid`, `include_parsed`, `include_context` | raw strings (+ parsed, + contexts) |
| `POST` | `/parse/references` | `llm-tasks` or `default` | `{"references": [...]}`; `parser=llm\|grobid` | structured records |
| `POST` | `/link/references` | `linking` | `{"reference": {"full_title": "...", "authors": ["Smith"], "publication_year": 2024}}` or object array with `batched=true`; `target=openalex\|matilda\|wikidata\|opencitations\|all` | target IDs and DOIs (per-source `error` on failure) |
| `POST` | `/process/references/grobid` | `default` | multipart PDF; `include_context` | raw + parsed references (+ contexts) |
| `POST` | `/classify/citations` | `default` | `{"citations": [{"section": "...", "citation": "..."}]}`; `mode=WS\|WoS\|M` | one intent label per sentence |
| `GET` | `/jobs/{job_id}/status` | — | job ID | status and error metadata |
| `GET` | `/jobs/{job_id}` | — | completed job ID | persisted result |

Interactive OpenAPI documentation is available at [`http://localhost:8001/docs`](http://localhost:8001/docs); the generated spec is `api_0.4.2.yml`.